In [1]:
# TEST DI SANITÀ: Verifica Data Leakage nella Normalizzazione

import numpy as np
from sklearn.model_selection import train_test_split
from data_handler.data_loader import normalize, load_cup, load_monk

print("="*70)
print("TEST 1: Verifica Data Leakage con Dataset CUP")
print("="*70)

# Carica dataset (usa il tuo path)
X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)

# Statistiche PRIMA della normalizzazione
print(f"\n📊 DATASET ORIGINALE (prima di split e normalizzazione)")
print(f"   X shape: {X.shape}")
print(f"   X mean (feature 0): {X[:, 0].mean():.6f}")
print(f"   X std (feature 0):  {X[:, 0].std():.6f}")
print(f"   X min/max: [{X.min():.3f}, {X.max():.3f}]")

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

print(f"\n📦 DOPO SPLIT (ancora non normalizzato)")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape:   {X_val.shape}")
print(f"   X_train mean (feature 0): {X_train[:, 0].mean():.6f}")
print(f"   X_val mean (feature 0):   {X_val[:, 0].mean():.6f}")

# Normalizzazione CORRETTA (fit su train, transform su val)
X_train_norm, mean_x, std_x = normalize(X_train)
X_val_norm, _, _ = normalize(X_val, mean=mean_x, std=std_x)

print(f"\n✅ DOPO NORMALIZZAZIONE CORRETTA")
print(f"   Statistiche usate (da X_train):")
print(f"      mean[0]: {mean_x[0]:.6f}")
print(f"      std[0]:  {std_x[0]:.6f}")
print(f"\n   X_train_norm (dovrebbe essere ~0 mean, ~1 std):")
print(f"      mean[0]: {X_train_norm[:, 0].mean():.6f}")
print(f"      std[0]:  {X_train_norm[:, 0].std():.6f}")
print(f"\n   X_val_norm (NON dovrebbe essere ~0 mean se fatto correttamente):")
print(f"      mean[0]: {X_val_norm[:, 0].mean():.6f}")
print(f"      std[0]:  {X_val_norm[:, 0].std():.6f}")

# Interpretazione
val_mean_diff = abs(X_val_norm[:, 0].mean())
if val_mean_diff < 0.01:
    print(f"\n⚠️  WARNING: X_val mean troppo vicino a 0 ({val_mean_diff:.6f})")
    print(f"   Possibile DATA LEAKAGE!")
else:
    print(f"\n✅ OK: X_val mean sufficientemente diverso da 0 ({val_mean_diff:.6f})")
    print(f"   Nessun data leakage rilevato.")

# -------------------------------------------------------------------
print("\n" + "="*70)
print("TEST 2: Verifica ERRATA (con data leakage - per confronto)")
print("="*70)

# Reload fresh data
X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)

# ERRORE: normalizza PRIMA dello split
X_wrong, mean_wrong, std_wrong = normalize(X)
X_train_wrong, X_val_wrong = train_test_split(X_wrong, test_size=0.2, random_state=42)

print(f"\n❌ NORMALIZZAZIONE ERRATA (prima dello split)")
print(f"   X_train_wrong mean[0]: {X_train_wrong[:, 0].mean():.6f}")
print(f"   X_val_wrong mean[0]:   {X_val_wrong[:, 0].mean():.6f}")
print(f"   ⚠️  Entrambi ~0 → DATA LEAKAGE PRESENTE!")

# -------------------------------------------------------------------
print("\n" + "="*70)
print("TEST 3: Verifica K-Fold (simulazione di 1 fold)")
print("="*70)

from sklearn.model_selection import KFold

X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

train_idx, val_idx = next(kf.split(X))
X_train_kf = X[train_idx]
X_val_kf = X[val_idx]

# Normalizzazione per-fold
X_train_kf_norm, mean_kf, std_kf = normalize(X_train_kf)
X_val_kf_norm, _, _ = normalize(X_val_kf, mean=mean_kf, std=std_kf)

print(f"\n✅ K-FOLD FOLD 1")
print(f"   X_train_kf_norm mean[0]: {X_train_kf_norm[:, 0].mean():.6f}")
print(f"   X_val_kf_norm mean[0]:   {X_val_kf_norm[:, 0].mean():.6f}")
print(f"   Stats from fold train mean[0]: {mean_kf[0]:.6f}")

# -------------------------------------------------------------------
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("✅ Se X_val mean ≠ 0 → Normalizzazione corretta (no leakage)")
print("❌ Se X_val mean ≈ 0 → Data leakage (statistiche calcolate su tutto il dataset)")
print("="*70)


TEST 1: Verifica Data Leakage con Dataset CUP

📊 DATASET ORIGINALE (prima di split e normalizzazione)
   X shape: (500, 12)
   X mean (feature 0): 2.192591
   X std (feature 0):  10.276026
   X min/max: [-22.966, 29.436]

📦 DOPO SPLIT (ancora non normalizzato)
   X_train shape: (400, 12)
   X_val shape:   (100, 12)
   X_train mean (feature 0): 2.314169
   X_val mean (feature 0):   1.706282

✅ DOPO NORMALIZZAZIONE CORRETTA
   Statistiche usate (da X_train):
      mean[0]: 2.314169
      std[0]:  10.227198

   X_train_norm (dovrebbe essere ~0 mean, ~1 std):
      mean[0]: -0.000000
      std[0]:  1.000000

   X_val_norm (NON dovrebbe essere ~0 mean se fatto correttamente):
      mean[0]: -0.059438
      std[0]:  1.022268

✅ OK: X_val mean sufficientemente diverso da 0 (0.059438)
   Nessun data leakage rilevato.

TEST 2: Verifica ERRATA (con data leakage - per confronto)

❌ NORMALIZZAZIONE ERRATA (prima dello split)
   X_train_wrong mean[0]: 0.011831
   X_val_wrong mean[0]:   -0.047325
  

In [1]:
# TEST: Verifica se set_state() causa problemi con optimizer

import numpy as np
from nn.model import Model
from nn.layers import Dense
from nn.activations import ReLU, Sigmoid
from nn.losses import BinaryCrossEntropy
from nn.optim import Adam
from training.trainer import Trainer

# 1. Crea modello semplice
np.random.seed(42)
model = Model(
    modules=[
        Dense(10, 5, seed=42),
        ReLU(),
        Dense(5, 1, seed=42),
        Sigmoid()
    ],
    loss=BinaryCrossEntropy(),
    optimizer=Adam(lr=0.01)
)

# 2. Simula training
X_train = np.random.randn(100, 10)
y_train = np.random.randint(0, 2, (100, 1))

trainer = Trainer(model, verbose=0)

# Train 20 epoche
history = trainer.fit(X_train, y_train, epochs=20, batch_size=32)

# 3. Salva stato epoca 20
state_epoch_20 = model.get_state()

# Stampa stato optimizer PRIMA del restore
print("=" * 70)
print("PRIMA del restore (epoca 20):")
print(f"   Adam timestep (t): {model.optimizer.t}")
print(f"   Numero di parametri tracciati: {len(model.optimizer.m)}")
first_param_id = next(iter(model.optimizer.m.keys()))
print(f"   m[primo_param] mean: {model.optimizer.m[first_param_id].mean():.6f}")
print(f"   v[primo_param] mean: {model.optimizer.v[first_param_id].mean():.6f}")

# 4. Train altre 30 epoche
history2 = trainer.fit(X_train, y_train, epochs=30, batch_size=32)

print("\n" + "=" * 70)
print("DOPO 30 epoche aggiuntive (totale epoca 50):")
print(f"   Adam timestep (t): {model.optimizer.t}")
print(f"   m[primo_param] mean: {model.optimizer.m[first_param_id].mean():.6f}")
print(f"   v[primo_param] mean: {model.optimizer.v[first_param_id].mean():.6f}")

# 5. Restore stato epoca 20 (simula early stopping)
model.set_state(state_epoch_20)

print("\n" + "=" * 70)
print("DOPO set_state() - PROBLEMA CRITICO:")
print(f"   Pesi ripristinati: epoca 20 ✅")
print(f"   Adam timestep (t): {model.optimizer.t}")  # DOVREBBE essere 20, non 50!
print(f"   m[primo_param] mean: {model.optimizer.m[first_param_id].mean():.6f}")
print(f"   ⚠️  Optimizer state NON ripristinato!")

# 6. Verifica impatto su next step
y_pred = model.forward(X_train[:1], training=True)
loss = model.compute_loss(y_train[:1], y_pred)
dY = model.loss.backward(y_pred, y_train[:1])
model.backward(dY)

# Guarda la magnitude del gradient step
first_param = next(model.parameters())
grad_magnitude = np.abs(first_param).mean()

print(f"\n   Gradient magnitude prima di step: {grad_magnitude:.6f}")

model.step()

first_param_after = next(model.parameters())
update_magnitude = np.abs(first_param - first_param_after).mean()

print(f"   Update magnitude dopo step: {update_magnitude:.6f}")
print(f"   Ratio (update/param): {update_magnitude/grad_magnitude:.6f}")

print("\n" + "=" * 70)
print("INTERPRETAZIONE:")
print("   Se ratio >> 1 → Update troppo grande → Possibile divergenza")
print("=" * 70)


PRIMA del restore (epoca 20):
   Adam timestep (t): 80
   Numero di parametri tracciati: 4
   m[primo_param] mean: -0.001041
   v[primo_param] mean: 0.000165

DOPO 30 epoche aggiuntive (totale epoca 50):
   Adam timestep (t): 200
   m[primo_param] mean: -0.001656
   v[primo_param] mean: 0.000445

DOPO set_state() - PROBLEMA CRITICO:
   Pesi ripristinati: epoca 20 ✅
   Adam timestep (t): 80
   m[primo_param] mean: -0.001041
   ⚠️  Optimizer state NON ripristinato!

   Gradient magnitude prima di step: 0.280202
   Update magnitude dopo step: 0.000000
   Ratio (update/param): 0.000000

INTERPRETAZIONE:
   Se ratio >> 1 → Update troppo grande → Possibile divergenza


In [3]:
# ============================================================================
# TEST SEMPLICE: CUP Dataset con MEE
# ============================================================================

from data_handler.data_loader import load_cup
from nn.model import Model
from nn.layers import Dense
from nn.activations import Tanh, ReLU, Identity
from nn.dropout import Dropout
from nn.losses import MEE, MSE
from nn.optim import Adam, SGD
from nn.metrics import MSE as MSE_Metric, MEE as MEE_Metric
from training.trainer import Trainer
from training.holdout_cv import holdout_validation
import numpy as np

# --- LOAD DATA ---
X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"X range: [{X.min():.2f}, {X.max():.2f}]")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

# --- BUILD MODEL ---
model = Model(
    modules=[
        Dense(12, 20, seed=42),  # Input: 12 features
        Tanh(),
        Dense(20, 10, seed=42),
        Tanh(),
        Dense(10, 4, seed=42),
        Identity(),   # Output: 4 targets (NO activation finale!)
    ],
    loss=MEE(),                  # Mean Euclidean Error
    optimizer=Adam(lr=0.001),
    metrics=[MEE_Metric(), MSE_Metric()],
)

print("\n" + "="*70)
print("MODEL ARCHITECTURE:")
print(model)
print("="*70)

# --- HOLDOUT VALIDATION ---
print("\n[1] Running Holdout Validation (80/20 split)...")
trainer = Trainer(model, verbose=0)

train_metrics, val_metrics, history = holdout_validation(
    X=X,
    y=y,
    model=model,
    trainer=trainer,
    val_split=0.2,
    stratified=False,        # Regressione: no stratification
    epochs=200,
    batch_size=32,
    shuffle=True,
    seed=42,
    verbose=1,
    normalize_data=True,     # ✅ IMPORTANTE per regressione
    normalize_target=True,   # ✅ IMPORTANTE per regressione
)

print("\n" + "="*70)
print("HOLDOUT VALIDATION RESULTS:")
print("="*70)
print(f"Train metrics: {train_metrics}")
print(f"Val metrics:   {val_metrics}")
print("="*70)

# --- TRAIN ON FULL DATASET ---
print("\n[2] Training on Full Dataset...")
model.reset()

trainer = Trainer(model, verbose=1)
history_full = trainer.fit(
    X, y,
    epochs=200,
    batch_size=32,
    shuffle=True,
    seed=42,
)

# --- FINAL EVALUATION ---
print("\n" + "="*70)
print("FINAL TRAIN METRICS:")
print("="*70)
final_metrics = trainer.evaluate(X, y)
print(f"MEE:  {final_metrics['MEE']:.6f}")
print(f"MSE:  {final_metrics['MSE']:.6f}")
print(f"Loss: {final_metrics['loss']:.6f}")
print("="*70)

# --- SANITY CHECK: Predict on a few samples ---
print("\n[3] Sanity Check - Predictions on first 3 samples:")
X_sample = X[:3]
y_sample = y[:3]
y_pred = model.predict_proba(X_sample)

for i in range(3):
    print(f"\nSample {i+1}:")
    print(f"  True:      {y_sample[i]}")
    print(f"  Predicted: {y_pred[i]}")
    print(f"  Error:     {np.linalg.norm(y_pred[i] - y_sample[i]):.6f}")


Dataset shape: X=(500, 12), y=(500, 4)
X range: [-22.97, 29.44]
y range: [-61.26, 59.35]

MODEL ARCHITECTURE:
Model(
  (0) Dense
  (1) Tanh
  (2) Dense
  (3) Tanh
  (4) Dense
  (5) Identity
)

[1] Running Holdout Validation (80/20 split)...
[Holdout] X Normalized. Train Mean[0]=2.314
[Holdout] y Normalized. Train Mean[0]=0.447

Starting Holdout Validation
Train samples: 400, Val samples: 100

HOLDOUT VALIDATION RESULTS:
Train metrics: {'loss': 1.2573140321583585, 'MEE': 1.2573140321583585, 'MSE': 1.9655672884502877}
Val metrics:   {'loss': 1.365593349912437, 'MEE': 1.365593349912437, 'MSE': 2.3813061954864416}

[2] Training on Full Dataset...

      _____     ___                 _ _ _ 
      \_   \   / __\__ ___   ____ _| | (_)
       / /\/  / /  / _` \ \ / / _` | | | |
    /\/ /_   / /__| (_| |\ V / (_| | | | |
    \____/   \____/\__,_| \_/ \__,_|_|_|_|
    
Epoch 1/200 [>.............................] loss: 35.6666 MEE: 35.6666 MSE: 1505.4898Epoch 2/200 [>............................